In [1]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_5"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [2]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)

Week-01

In [3]:
X_init = data_in
y_init = data_out

kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# Surrogate to maximise (negative for minimize)
def surrogate_neg(x):
    return -gp.predict(x.reshape(1, -1))[0]

# Bounds for normalized inputs
bounds = [(0,1), (0,1), (0,1), (0,1)]

# Try multiple random starts to avoid local issues
best_x = None
best_val = float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x
print("Next point to evaluate:", x_next)


Next point to evaluate: [0.96208674 0.13076833 0.84922277 0.78444212]


Week-02

In [4]:
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464]
])

new_outputs = np.array([
    1091.3271430129832
])


# Combine all data
X_all = np.vstack([data_in, new_points])
y_all = np.concatenate([data_out, new_outputs])

# --- Refit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_all)

# --- Generate candidate next points around current best ---
best_x = new_points
sigma = 0.02  # tweak size for local exploration
num_candidates = 5

x_next_candidates = best_x + np.random.normal(0, sigma, size=(num_candidates, 4))
# Ensure all points are within [0,1]
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("Candidate next points to evaluate:")
print(x_next_candidates)

Candidate next points to evaluate:
[[0.26214374 0.83959504 0.86903359 0.87632395]
 [0.23919593 0.8412031  0.90083356 0.89077154]
 [0.24899377 0.83830642 0.85109944 0.90115073]
 [0.23967367 0.8359426  0.89674444 0.8925783 ]
 [0.17300392 0.83484132 0.88951819 0.86762306]]


In [5]:
# --- Add the two new data points ---
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464],
    [0.245154,0.843081,0.898729,0.883391]
])

new_outputs = np.array([
    1091.3271430129832,
    1195.7279770056589
])

# Combine all data
X_all = np.vstack([data_in, new_points])
y_all = np.concatenate([data_out, new_outputs])

# --- Fit updated Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-8,                # smaller noise term for precision
    normalize_y=True,
    n_restarts_optimizer=5     # more robust kernel fitting
)
gp.fit(X_all, y_all)

# --- Define acquisition function (UCB variant) ---
def surrogate_neg_ucb(x, kappa=2.0):
    mean, std = gp.predict(x.reshape(1, -1), return_std=True)
    return -(mean + kappa * std)

# --- Search bounds for normalized inputs ---
bounds = [(0, 1), (0, 1), (0, 1), (0, 1)]

# --- Optimize acquisition function for next sampling point ---
best_x, best_val = None, float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg_ucb, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x

print("Suggested next point to evaluate:", x_next)

# --- Optionally: generate a few local perturbations for fine exploration ---
sigma = 0.01
num_candidates = 5
x_next_candidates = x_next + np.random.normal(0, sigma, size=(num_candidates, 4))
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("\nLocal candidate points for fine-tuning:")
print(x_next_candidates)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)

x_next_6dp = np.round(x_next, 6)
x_next_6dp

Suggested next point to evaluate: [0.45006011 0.77435438 0.03786367 0.80667699]

Local candidate points for fine-tuning:
[[0.4551899  0.76252456 0.02036268 0.79657138]
 [0.43853756 0.77323692 0.03036472 0.80760767]
 [0.45822892 0.77869227 0.02912416 0.80758994]
 [0.44985078 0.76629923 0.0421556  0.80322206]
 [0.45829802 0.77478126 0.0411306  0.80875643]]
0.450060-0.774354-0.037864-0.806677


array([0.45006 , 0.774354, 0.037864, 0.806677])

week-04

In [6]:
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464],
    [0.245154,0.843081,0.898729,0.883391],
    [0.237286,0.897828,0.947445,0.897134]
])

new_outputs = np.array([
    1091.3271430129832,
    1195.7279770056589,
    1880.5766271350337
])


# Combine all data
X_all = np.vstack([data_in, new_points])
y_all = np.concatenate([data_out, new_outputs])


eps= 1e-20
signs = np.sign(y_all)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_all) + eps)


kernel = Matern(length_scale = 0.1, nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_trans)

def acquisition_ei(X, gp, y_best, xi=0.05):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.03
n = 6

# Create 1D ranges
x1 = np.linspace(margin, 1-margin, n)
x2 = np.linspace(margin, 1-margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)

# Create 4D meshgrid
X1, X2, X3, X4 = np.meshgrid(
    x1, x2, x3, x4, 
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel()
]).T

#bounds = [(X_all[:,i].min(), X_all[:,i].max()) for i in range(4)]
#num_candidates = 5000
#X_candidates = np.column_stack([
#    np.random.uniform(b[0], b[1], num_candidates) for b in bounds
#])

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=1.0)

# --- 7. Select the next point ---
next = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"[{next[0]:.6f}, {next[1]:.6f}, {next[2]:.6f}, {next[3]:.6f}]")

print(cf.format_inputdata(next))

[0.594000, 0.970000, 0.970000, 0.970000]
0.594000-0.970000-0.970000-0.970000


[0.406000, 0.970000, 0.970000, 0.970000] for 0.05 <br/>
[0.782000, 0.970000, 0.970000, 0.970000] for 1.0 <br/>
[0.782000, 0.970000, 0.970000, 0.970000] for 2.0 <br/>